# Part 2 — Longitudinal Trajectories

**Goal:** Visualise how each subject's Centiloid value changes over time.

A "trajectory" is the sequence of PET measurements for one person, plotted against years since their first scan. By overlaying all subjects you can see the range of progression patterns — some people accumulate amyloid rapidly, others stay flat for years.

---

**Run cells in order from top to bottom using Shift + Enter.**  
When you finish a TODO, **delete the `raise NotImplementedError(...)` line** in that cell before moving on.

---

## What's pre-built for you

The data loading from Part 1 has been packaged into a helper function `_load_pib_data()` defined in the Setup cell below — you don't need to rewrite those steps. A new helper is also available:

| Function | What it does |
|----------|-------------|
| `get_multi_sample_subjects(df, subject_col)` | Returns a list of subject IDs that have ≥ 2 measurements — required to draw a trajectory line |

Your tasks are three TODOs focused entirely on plotting.

---

> **Jupyter tip — upstream errors:** If a cell throws an unexpected error, the problem is often in an earlier cell (e.g. a variable was overwritten). Re-run all cells from the top using **Run All** (the ▶▶ button in the toolbar) to reset the notebook state, then try again.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

from utils.oasis_utils import (
    load_dataset,
    validate_columns,
    add_extracted_subject_id,
    add_days_since_start_from_id,
    get_multi_sample_subjects,
    ensure_dir,
)

# ── Configuration ─────────────────────────────────────────────────────────────
DATA_PATH   = os.path.join("data", "OASIS3_PUP.xlsx")
RESULTS_DIR = "results"

TRACER   = "PIB"
ID_COL   = "PUP_PUPTIMECOURSEDATA ID"
SUVR_COL = "Centil_fSUVR_TOT_CORTMEAN"

ensure_dir(RESULTS_DIR)


def _load_pib_data():
    """Load, filter, and time-stamp the PIB measurements. (Pre-built — do not modify.)"""
    df = load_dataset(DATA_PATH)
    validate_columns(df, [ID_COL, SUVR_COL, "tracer"])
    df = df[df["tracer"] == TRACER].copy()
    df = df.dropna(subset=[ID_COL, SUVR_COL])
    df = add_extracted_subject_id(df, ID_COL)
    df = add_days_since_start_from_id(df, ID_COL, output_days="Days", output_years="Years")
    return df


# Load data and filter to subjects with ≥2 scans
df = _load_pib_data()
longitudinal_ids = get_multi_sample_subjects(df, "Subject_ID_Extracted")
df_long = df[df["Subject_ID_Extracted"].isin(longitudinal_ids)].copy()

print(f"Subjects with ≥2 scans: {len(longitudinal_ids)}")
df_long.head()

---
## Grouping the data — run this cell before plotting

The cell below loops over subjects and prints the first group so you can see the structure before writing your plotting code. Just run it — no changes needed.

> **Important:** Never write `df_long = df_long.groupby(...)`. That would replace the DataFrame with a GroupBy object, and re-running the cell would then cause an `AttributeError`. The `for` loop pattern below keeps `df_long` intact.

In [ ]:
for subject_id, group in df_long.groupby('Subject_ID_Extracted'):
    group = group.sort_values('Years')
    print(group[['Subject_ID_Extracted', 'Years', 'Centil_fSUVR_TOT_CORTMEAN']].head())
    break  # prints just the first subject — remove to see all

---
## TODO 1 — Plot all subject trajectories

Inside a `groupby` loop, for each sorted group call `plt.plot()` and `plt.scatter()` to draw a line and mark the individual measurements.

Use these style parameters so all background subjects look consistent:
```python
plt.plot(   ..., color='steelblue', alpha=0.25, linewidth=1.0)
plt.scatter(..., color='steelblue', alpha=0.35, s=18)
```

Call `plt.show()` at the end. You should see many overlapping blue lines — that's correct.

In [ ]:
plt.figure(figsize=(10, 6))

# YOUR CODE HERE

raise NotImplementedError("TODO 1: plot all subject trajectories")

plt.show()

---
## TODO 2 — Highlight one example subject

With so many overlapping lines it's hard to follow any single person. Pick one subject and re-plot their trajectory on top in a contrasting colour.

**Steps:**

**a)** Choose a subject ID to highlight. To pick the one with the most measurements:
```python
example_id = df_long.groupby('Subject_ID_Extracted').size().idxmax()
```
Or just use `longitudinal_ids[0]` to pick the first one.

**b)** Filter `df_long` to just that subject and sort by `'Years'`:
```python
example = df_long[df_long['Subject_ID_Extracted'] == example_id].sort_values('Years')
```

**c)** Plot their line and points on top of the existing figure using `plt.plot()` and `plt.scatter()`.  
Use `color='crimson'` and a `label` so they appear in the legend.

> **Note:** `plt.plot()` and `plt.scatter()` add to the most recently created figure — as long as you haven't called `plt.show()` yet, this cell will draw on top of the plot from TODO 1b.

In [ ]:
# YOUR CODE HERE

raise NotImplementedError("TODO 2: highlight one example subject")

---
## TODO 3 — Finalise and save the figure

Add labels, a title, and clean styling, then save.

**a)** Set axis labels and title:
```python
plt.xlabel("Years Since First Scan", fontsize=13)
plt.ylabel("Centiloid (non-RSF)", fontsize=13)
plt.title("Longitudinal PIB Centiloid Trajectories", fontsize=14, fontweight='bold')
```

**b)** Add a legend and a light grid:
```python
plt.legend(frameon=False)
plt.grid(True, linestyle='--', alpha=0.2)
```

**c)** Remove the top and right spines:
```python
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
```

**d)** Save the figure:
```python
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "trajectories.png"), dpi=200)
plt.show()
```

In [ ]:
# YOUR CODE HERE

raise NotImplementedError("TODO 3: finalise and save the figure")

---
## Reflection questions

Answer in a new markdown cell below (double-click to edit, **Shift + Enter** to render).

**Q1.** Most subjects show relatively flat trajectories. A few show steep increases. What biological differences might explain this?

**Q2.** Some subjects start with already-high Centiloid values (> 50) and others start near zero. What could explain these baseline differences?

**Q3.** Does the rate of accumulation appear constant over time for most subjects, or does it seem to accelerate or decelerate? What might that imply about the biology of amyloid deposition?